In [1]:
import os
import openai
import json

from dotenv import load_dotenv
from system_prompts import ACTOR_PROMPT
from system_prompts import GRADER_PROMPT

load_dotenv(dotenv_path=".env.local")

# Set your API key
api_key = os.environ.get("OPENAI_API_KEY") 

Successfully downloaded prompt from Google doc.
Successfully downloaded prompt from Google doc.


## Initialize the Actor and Grader

In [2]:
# Initialize the actor_llm
actor_llm = openai.OpenAI(api_key=api_key)

In [18]:
grader_llm = openai.OpenAI(api_key=api_key)


## Introduce the Actor

In [3]:
print("Starting chat with OpenAI (type 'exit' to quit)")
messages = [{"role": "system", "content": ACTOR_PROMPT}]
actor_response = actor_llm.responses.create(
    model="gpt-3.5-turbo",
    input=messages,
    stream=True,
    )
print(actor_response)

Starting chat with OpenAI (type 'exit' to quit)


In [ ]:
for event in actor_response:
    event_type = getattr(event, 'type', '')

    if event_type == 'response.output_text.delta':
        delta = getattr(event, 'delta', '')
        yield delta

In [ ]:
def extract_completed_from_stream(events):
    """
    Extract the full text response from a stream of OpenAI API events.
    
    Args:
        events: List of event dictionaries from the OpenAI stream
        
    Returns:
        The complete text response
    """
    full_text = ""
    
    for event in events:
        event_type = getattr(event, 'type', '')
        
        # Method 1: Accumulate delta chunks
        if event_type == 'response.output_text.delta':
            delta = getattr(event, 'delta', '')
            full_text += delta

        # Method 2: Get from done event (alternative approach)
        elif event_type == 'response.completed':
            return event.response

    # return full_text

In [ ]:
def extract_response_from_stream(events) -> str:
    """
    Extract the full text response from a stream of OpenAI API events.
    
    Args:
        events: List of event dictionaries from the OpenAI stream
        
    Returns:
        The complete text response
    """
    full_text = ""
    
    for event in events:
        event_type = getattr(event, 'type', '')
        
        # Method 1: Accumulate delta chunks
        if event_type == 'response.output_text.delta':
            delta = getattr(event, 'delta', '')
            full_text += delta

        # Method 2: Get from done event (alternative approach)
        elif event_type == 'response.completed':
            return event.response

    # return full_text

In [26]:
response = extract_response_from_stream(actor_response)

In [28]:
response.id

'resp_680ac43f89348191bbc3b4117e6420e10451c8cdd9f6e0bf'

In [ ]:
for event in actor_response:
    print(event)

ResponseCreatedEvent(response=Response(id='resp_680ac246c3b48191817e37ca5c46f8080b45c5f3f28c736b', created_at=1745535558.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-3.5-turbo-0125', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, max_output_tokens=None, previous_response_id=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), service_tier='auto', status='in_progress', text=ResponseTextConfig(format=ResponseFormatText(type='text')), truncation='disabled', usage=None, user=None, store=True), type='response.created')
ResponseInProgressEvent(response=Response(id='resp_680ac246c3b48191817e37ca5c46f8080b45c5f3f28c736b', created_at=1745535558.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-3.5-turbo-0125', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, max_output_to

In [ ]:
actor_response.output_text

'Hi, I’m Ashley. How’s your first week going?'

In [6]:
chat_id = actor_response.id
chat_id

'chatcmpl-BPwlL2bc4RPRCPyJ7LoxwHN6pxKV3'

In [7]:
messages = [
    {"role": "user",
     "content": "User: It's going well, I'm having a bit of a hard time getting past the learning curve for the Microsoft suit, but otherwise so far so good! How are you?"
    }]
actor_response = actor_llm.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    )

In [8]:
actor_response.choices[0].message.content

"I'm glad to hear things are going well for you! The Microsoft Suite can be a bit tricky at first, but with practice and patience, you'll get the hang of it. As for me, I'm just here ready to assist you with anything you need help with. Let me know if you have any questions or need guidance with anything!"

## Introduce the grader

In [13]:
print("Starting chat with OpenAI (type 'exit' to quit)")
messages = [{"role": "system", "content": GRADER_PROMPT}]
grader_response = grader_llm.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    response_format={ "type": "json_object" }
    )
grader_content = grader_response.choices[0].message.content
grader_json = json.loads(grader_content)
print(json.dumps(grader_json, indent=4))

Starting chat with OpenAI (type 'exit' to quit)
{
    "score": {
        "open_ended_question": 0,
        "follow_up_question": 0,
        "empathy": 0,
        "paraphrasing": 0,
        "unsolicited_advice": 0
    },
    "feedback": "Try asking more open-ended questions to encourage Ashley to share more. Show empathy to connect on a deeper level.",
    "level_token": "LEVEL1",
    "level_description": "The user is asking surface-level or closed-ended questions, not building much rapport or safety. The actor (Ashley) should only share basic, non-vulnerable information."
}


In [10]:
grader_response

ChatCompletion(id='chatcmpl-BPwlki3n9DFPihlKLpvqwoPiZZZRF', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "score": {\n    "open_ended_question": 0,\n    "follow_up_question": 0,\n    "empathy": 0,\n    "paraphrasing": 0,\n    "unsolicited_advice": 0\n  },\n  "feedback": "To improve, try asking open-ended questions to encourage Ashley to share more.",\n  "level_token": "LEVEL1",\n  "level_description": "The user is asking surface-level or closed-ended questions, not building much rapport or safety. The actor (Ashley) should only share basic, non-vulnerable information."\n}', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1745523808, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=126, prompt_tokens=686, total_tokens=812, completion_tokens_details=CompletionTokensDetails(acce

In [16]:
grader_json["level_token"]

'LEVEL1'

## First Message w/ Actor + Grader

In [ ]:
messages = [
    {"role": "user",
     "content": (
         "User: User: Oh yeah, do you have a lot on your plate today?"
     )
    }]
actor_response = actor_llm.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    stream=True
    )